# XAI Visualization with `contextualshap` and `DiCE`

This notebook provides a way to generate SHAP (and visualizations) and DiCE counterfactuals given a dataset and model.
The primary purpose of this notebook is to generate explanations to better understand how the model generates outcomes
of predictions given a dataset, with the primary purpose of making AI models and the associated workflows more practical
for real use cases.

This notebook uses PyTorch and lightning to build a DNN training pipeline and generate SHAP values with contextualshap and counter factual
with DiCE.

## Loading the Dataset

This example uses Kaggle to retrieve the `pima-indians-diabeter` dataset and load them into Pandas DataFrame.
A few parameters (constants) must be filled like `continuous_features` and `outcome_name` which refer to columns of continuous features and column
that contain outcome values (must be binary 1 or 0).

### Kaggle

The most common way to load datasets from Kaggle is using `kagglehub` package. For example:

```python
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Download latest version
dataset = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "uciml/pima-indians-diabetes-database", 'diabetes.csv')
```

In [ ]:
import kagglehub
from fontTools.misc.cython import returns
from kagglehub import KaggleDatasetAdapter

# Download latest version
dataset = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "alexteboul/diabetes-health-indicators-dataset", 'diabetes_binary_health_indicators_BRFSS2015.csv')

### General Dataset and Training Settings

To use DiCE and many of this notebook automatic transformations (such as one-hot encoding), columns that are continuous must be
separated. You must supply which columns are considered continuous by setting `continuous_features` below. The `outcome_name`
defines which column is the outcome, currently must only be 0 or 1.

Learning rate is automatically determined through PyTorch Lightning LR finder. We also provide both DNN and CNN models for comparison.

In [ ]:
continuous_features = ['BMI', 'MentHlth', 'PhysHlth', 'Age']
outcome_name = 'Diabetes_binary'
learning_rate = 1e-4
max_epochs = 40
samples_to_find_cfs = 5
cfs_per_sample = 4
samples_for_shap_train = 500
samples_for_shap_calculate = 50
load_model_from_checkpoint = True
openai_api_key = '' # Must be filled to use contextualshap

In [ ]:
# Create DiCE dataset instance
import dice_ml

d = dice_ml.Data(dataframe=dataset, continuous_features=continuous_features, outcome_name=outcome_name)

In [ ]:
# Split the dataset into train and test
from sklearn.model_selection import train_test_split

target = dataset[outcome_name]
train_val_dataset, test_dataset, train_val_target, _ = train_test_split(dataset, target, test_size=0.2, random_state=0, stratify=target)
train_dataset, val_dataset, _, _ = train_test_split(train_val_dataset, train_val_target, test_size=0.2, random_state=0, stratify=train_val_target)

## Building the Model

To ease things up we use DiCE utilities such as `ohe_min_max_transformation` to convert categorical features into One-Hot Encoded features.
This increases the number of columns in the dataset. In this example, a tiny lightning module is created with standard DNN
model implemented with PyTorch.

In [ ]:
# Building the PyTorch model
from dice_ml.utils.helpers import ohe_min_max_transformation, inverse_ohe_min_max_transformation
from sklearn.preprocessing import FunctionTransformer

trans = FunctionTransformer(func=ohe_min_max_transformation, inverse_func=inverse_ohe_min_max_transformation, check_inverse=False, validate=False, kw_args={'data_interface': d}, inv_kw_args={'data_interface': d})
train_data = trans.transform(train_dataset)
val_data = trans.transform(val_dataset)

feature_len = len(train_data.columns)-1

# device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device = 'cpu'

In [ ]:
import torch
from torch import nn

class DenseModel(nn.Module):
    def __init__(self):
        # Call the parent class constructor
        super(DenseModel, self).__init__()

        # Define the layers
        self.norm = nn.LayerNorm(feature_len)
        self.linear1 = nn.Linear(feature_len, 128)
        self.activation1 = nn.ReLU()
        self.linear2 = nn.Linear(128, 256)
        self.activation2 = nn.ReLU()
        self.linear3 = nn.Linear(256, 128)
        self.activation3 = nn.ReLU()
        self.linear4 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.norm(x)
        x = self.linear1(x)
        x = self.activation1(x)
        x = self.linear2(x)
        x = self.activation2(x)
        x = self.linear3(x)
        x = self.activation3(x)
        x = self.linear4(x)
        x = torch.sigmoid(x)
        return x

# Create an instance of the model
model = DenseModel()

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class ConvolutionalModel(nn.Module):
    def __init__(self):
        # Call the parent class constructor
        super(ConvolutionalModel, self).__init__()

        self.norm = nn.LayerNorm(feature_len)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=16, kernel_size=3)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.fc1 = nn.Linear(in_features=16*feature_len, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=1)

    def forward(self, x):
        x = self.norm(x)
        x = x.unsqueeze(1)
        x = self.conv1(x)
        x = self.pool1(F.relu(x))
        x = x.view(x.size(0), -1) # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        x = torch.sigmoid(x)
        return x

# Create an instance of the model
model = ConvolutionalModel()

In [ ]:
# Building the PyTorch Lightning module for training, validation and testing

import lightning.pytorch as L
import numpy as np

x = torch.from_numpy(train_data.drop(columns=outcome_name).to_numpy().astype(np.float32))
y = torch.from_numpy(np.expand_dims(train_data[outcome_name].to_numpy(), axis=-1).astype(np.float32))
x_val = torch.from_numpy(val_data.drop(columns=outcome_name).to_numpy().astype(np.float32))
y_val = torch.from_numpy(np.expand_dims(val_data[outcome_name].to_numpy(), axis=-1).astype(np.float32))

class FullModule(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = DenseModel()
        self.lr = 1e-4
        self.batch_size = 16

    def forward(self, inputs):
        return self.model(inputs)

    def training_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.forward(inputs)
        loss = torch.nn.functional.binary_cross_entropy(output, target)
        if batch_idx % 10 == 0:
            self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, target = batch
        output = self.forward(inputs)
        loss = torch.nn.functional.binary_cross_entropy(output, target)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        inputs, target = batch

        output = self.forward(inputs)
        loss = torch.nn.functional.binary_cross_entropy(output, target)
        predictions_direct = (output > 0.5).float()
        accuracy = (predictions_direct == target).float().mean()
        self.log_dict({"test_loss": loss, "test_acc": accuracy})

    def predict_step(self, batch, batch_idx):
        if isinstance(batch, tuple):
            inputs, _ = batch
        else:
            inputs = batch[0]

        predictions = self.forward(inputs)
        return predictions

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr, eps=1e-8, betas=(0.28, 0.93))
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss', # Must match the key used in self.log()
                'interval': 'epoch',
                'frequency': 1,
            }
        }

    def train_dataloader(self):
        return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x, y), num_workers=19, batch_size=self.batch_size)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_val, y_val), num_workers=19, batch_size=self.batch_size)


In [ ]:
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import TQDMProgressBar, LearningRateMonitor
from lightning.pytorch.tuner import Tuner
import os

checkpoint_path = "full_module.ckpt"

logger = TensorBoardLogger("tb_logs", name="dnn1")
trainer = L.Trainer(callbacks=[LearningRateMonitor(logging_interval='step')], max_epochs=max_epochs, enable_model_summary=True, logger=logger, enable_progress_bar=False)
# trainer = L.Trainer(callbacks=[TQDMProgressBar(), LearningRateMonitor(logging_interval='step')], max_epochs=30, enable_model_summary=True, logger=logger)
tuner = Tuner(trainer)
lm = FullModule()
if load_model_from_checkpoint and os.path.exists(checkpoint_path):
    lm = FullModule.load_from_checkpoint(checkpoint_path=checkpoint_path)

In [ ]:
# We use PyTorch Lightning's built-in learning rate finder to find a good learning rate'
%matplotlib notebook

lr_finder = tuner.lr_find(lm)
fig = lr_finder.plot(suggest=True)
fig.show()
new_lr = lr_finder.suggestion()
lm.hparams.lr = new_lr

In [ ]:
from lightning.pytorch.utilities import disable_possible_user_warnings
import logging
import numpy as np
import os

# For the new lightning package structure (v2.0+)
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

disable_possible_user_warnings()

if not load_model_from_checkpoint or (load_model_from_checkpoint and not os.path.exists(checkpoint_path)):
    trainer.fit(lm)
    trainer.save_checkpoint(checkpoint_path)

In [ ]:
test_data = trans.transform(test_dataset)
x_test = torch.from_numpy(test_data.drop(columns=outcome_name).to_numpy().astype(np.float32))
y_test = torch.from_numpy(np.expand_dims(test_data[outcome_name].to_numpy(), axis=-1).astype(np.float32))
test_dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), num_workers=19, batch_size=16)

trainer.test(model=lm, dataloaders=test_dl)

## DiCE

DiCE can accept PyTorch models (and other models as well). Modify the cell below if using other model.
Also modify the `query_instance` below to change which sample to generate counter factuals from. By default
it looks for one sample with outcome equals to 1, i.e. having diabetes in a diabetes dataset, and look for
its counterfactuals. At most 5 samples are taken. If finding 4 CF per sample, this will result in at most
20 CFs.

To automatically generate counterfactuals, we simply searched 5 samples that are **predicted** to have
outcome equals to true (> 0.5 prediction value). This way, we look for what the model thinks is diabetes.
The model inaccuracy might incorrectly predict some samples, which means that the outcome differs with the
original dataset. This ensures that all the samples we select will generate consistent counter factuals,
i.e. we will always look for CFs to generate `false` outcome given `true` samples, due to the fact that
DiCE uses the model to predict the outcome first and look for the CFs. We use the slower, and yet more accurate, `gradient` method
to generate counterfactuals. This however, still relies on the model being accurately trained. Inaccurate model
can result in also incorrect counterfactuals.

In [ ]:
test_instances = test_dataset.loc[test_dataset[outcome_name] == 1]
predict_results = trainer.predict(lm, torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.tensor(trans.transform(test_instances.drop(columns=outcome_name)).to_numpy().astype(np.float32))), batch_size=1))
test_instances.loc[:, outcome_name] = [ int(n > 0.5) for n in predict_results ]
query_instance = test_instances.loc[test_instances[outcome_name] > 0.5].drop(columns=outcome_name).sample(n=samples_to_find_cfs, random_state=123456)


In [ ]:
# DiCE explanation instance
m = dice_ml.Model(model=lm.model, backend="PYT", func="ohe-min-max")
exp = dice_ml.Dice(d, m, method="gradient")
# Generate counterfactual examples, without feature weights
dice_exp = exp.generate_counterfactuals(query_instance, total_CFs=cfs_per_sample, desired_class="opposite")
# Visualize counterfactual explanation
dice_exp.visualize_as_dataframe()

## SHAP

This notebook also presents SHAP values for the model. SHAP values are a powerful tool to explain the output of any machine learning model.
They can be used to understand how a machine learning model makes predictions and to explain why a particular prediction was made.
This notebook uses the `KernelExplainer` from `shap` package, and also uses `contextualshap` to adds contextual information from ChatGPT.

In total, 3 visualizations are presented:
1. SHAP values for the model output in beeswarm plot, representing SHAP values from a subset of the dataset
2. Mean SHAP values for the model output for a subset of the dataset, in waterfall plot
3. SHAP values for the model output for a subset of the dataset, with contextual information from ChatGPT, also with bar plot of mean shap values

In [ ]:
import shap
import pandas as pd

shap.initjs()

def f(x):
    x = trans.transform(pd.DataFrame(x, columns=train_dataset.drop(columns=outcome_name).columns)).values
    return lm.model(torch.tensor(np.astype(x, np.float32))).detach().numpy().flatten()


kernel_explainer = shap.KernelExplainer(f, train_dataset.drop(columns=outcome_name).sample(n=samples_for_shap_train), link="logit")
explanation = kernel_explainer(train_dataset.drop(columns=outcome_name).sample(n=samples_for_shap_calculate).to_numpy())

In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

mean_shap_values = explanation.abs.mean(axis=0)
feature_weights = {}

for i in range(len(explanation.feature_names)):
    feature_weights[explanation.feature_names[i]] = mean_shap_values[i].values.item()
shap.plots.beeswarm(explanation)
fig, ax = plt.subplots()
shap.plots.waterfall(mean_shap_values, show=False)
plt.show()

In [ ]:
if openai_api_key is not None and openai_api_key != '':
    import contextualshap.plots
    contextualshap.plots.bar(explanation, openai_api_key=openai_api_key)